# SC Dataset Exploration (Staging)

This notebook explores the South Carolina (SC) **staging** dataset to understand household counts, income distribution, and demographic characteristics.

In [1]:
from policyengine_us import Microsimulation
import pandas as pd
import numpy as np

SC_DATASET = "hf://policyengine/policyengine-us-data/staging/states/SC.h5"

In [2]:
# Load SC staging dataset
sim = Microsimulation(dataset=SC_DATASET)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


SC.h5:   0%|          | 0.00/38.1M [00:00<?, ?B/s]

In [3]:
# Check dataset size
household_weight = sim.calculate("household_weight", period=2025)
household_count = sim.calculate("household_count", period=2025, map_to="household")
person_count = sim.calculate("person_count", period=2025, map_to="household")

print(f"Number of households in dataset: {len(household_weight):,}")
print(f"Household count (weighted): {household_count.sum():,.0f}")
print(f"Person count (weighted): {person_count.sum():,.0f}")

Number of households in dataset: 22,528
Household count (weighted): 1,587,912
Person count (weighted): 4,788,894


In [4]:
# Check income distribution (weighted vs unweighted, household and person level)
agi_household = sim.calculate("adjusted_gross_income", period=2025, map_to="household")
agi_hh_array = np.array(agi_household)
hh_weights = np.array(sim.calculate("household_weight", period=2025))

agi_person = sim.calculate("adjusted_gross_income", period=2025, map_to="person")
agi_person_array = np.array(agi_person)
person_weights = np.array(sim.calculate("person_weight", period=2025))

# Weighted percentile calculation
def weighted_percentile(values, weights, percentile):
    sorted_indices = np.argsort(values)
    sorted_values = values[sorted_indices]
    sorted_weights = weights[sorted_indices]
    cumulative_weight = np.cumsum(sorted_weights)
    idx = np.searchsorted(cumulative_weight, cumulative_weight[-1] * percentile / 100)
    return sorted_values[min(idx, len(sorted_values)-1)]

# Unweighted medians
unweighted_median_hh = np.median(agi_hh_array)
unweighted_median_person = np.median(agi_person_array)

# Weighted medians
weighted_median_hh = weighted_percentile(agi_hh_array, hh_weights, 50)
weighted_median_person = weighted_percentile(agi_person_array, person_weights, 50)

# Weighted averages
weighted_avg_hh = np.average(agi_hh_array, weights=hh_weights)
weighted_avg_person = np.average(agi_person_array, weights=person_weights)

# Average household size
total_persons = person_weights.sum()
total_households = hh_weights.sum()
avg_hh_size = total_persons / total_households

print("=" * 60)
print("INCOME DISTRIBUTION SUMMARY")
print("=" * 60)
print(f"\nHousehold AGI:")
print(f"  Unweighted median: ${unweighted_median_hh:,.0f}")
print(f"  Weighted median:   ${weighted_median_hh:,.0f}")
print(f"  Weighted average:  ${weighted_avg_hh:,.0f}")

print(f"\nPerson AGI:")
print(f"  Unweighted median: ${unweighted_median_person:,.0f}")
print(f"  Weighted median:   ${weighted_median_person:,.0f}")
print(f"  Weighted average:  ${weighted_avg_person:,.0f}")

print(f"\nAverage household size: {avg_hh_size:.1f}")

print(f"\nWeighted household AGI percentiles:")
print(f"  25th percentile: ${weighted_percentile(agi_hh_array, hh_weights, 25):,.0f}")
print(f"  50th percentile: ${weighted_percentile(agi_hh_array, hh_weights, 50):,.0f}")
print(f"  75th percentile: ${weighted_percentile(agi_hh_array, hh_weights, 75):,.0f}")
print(f"  90th percentile: ${weighted_percentile(agi_hh_array, hh_weights, 90):,.0f}")
print(f"  95th percentile: ${weighted_percentile(agi_hh_array, hh_weights, 95):,.0f}")
print(f"  Max AGI: ${agi_hh_array.max():,.0f}")

INCOME DISTRIBUTION SUMMARY

Household AGI:
  Unweighted median: $67,907
  Weighted median:   $60,378
  Weighted average:  $105,617

Person AGI:
  Unweighted median: $67,460
  Weighted median:   $56,366
  Weighted average:  $100,583

Average household size: 3.0

Weighted household AGI percentiles:
  25th percentile: $27,117
  50th percentile: $60,378
  75th percentile: $105,515
  90th percentile: $153,279
  95th percentile: $237,856
  Max AGI: $353,653,602


In [5]:
# Check households with children
is_child = sim.calculate("is_child", period=2025, map_to="person")
household_id = sim.calculate("household_id", period=2025, map_to="person")
household_weight = sim.calculate("household_weight", period=2025, map_to="person")

# Create DataFrame
df_households = pd.DataFrame({
    'household_id': household_id,
    'is_child': is_child,
    'household_weight': household_weight
})

# Count children per household
children_per_household = df_households.groupby('household_id').agg({
    'is_child': 'sum',
    'household_weight': 'first'
}).reset_index()

# Calculate weighted household counts
total_households_with_children = children_per_household[children_per_household['is_child'] > 0]['household_weight'].sum()
households_with_1_child = children_per_household[children_per_household['is_child'] == 1]['household_weight'].sum()
households_with_2_children = children_per_household[children_per_household['is_child'] == 2]['household_weight'].sum()
households_with_3plus_children = children_per_household[children_per_household['is_child'] >= 3]['household_weight'].sum()

print(f"\nHouseholds with children (weighted):")
print(f"  Total households with children: {total_households_with_children:,.0f}")
print(f"  Households with 1 child: {households_with_1_child:,.0f}")
print(f"  Households with 2 children: {households_with_2_children:,.0f}")
print(f"  Households with 3+ children: {households_with_3plus_children:,.0f}")


Households with children (weighted):
  Total households with children: 637,558
  Households with 1 child: 308,153
  Households with 2 children: 190,106
  Households with 3+ children: 139,299


In [6]:
# Check children by age groups
df = pd.DataFrame({
    "household_id": sim.calculate("household_id", map_to="person"),
    "tax_unit_id": sim.calculate("tax_unit_id", map_to="person"),
    "person_id": sim.calculate("person_id", map_to="person"),
    "age": sim.calculate("age", map_to="person"),
    "person_weight": sim.calculate("person_weight", map_to="person")
})

# Filter for children and apply weights
children_under_18_df = df[df['age'] < 18]
children_under_6_df = df[df['age'] < 6]
children_under_3_df = df[df['age'] < 3]

# Calculate weighted totals
total_children = children_under_18_df['person_weight'].sum()
children_under_6 = children_under_6_df['person_weight'].sum()
children_under_3 = children_under_3_df['person_weight'].sum()

print(f"\nChildren by age:")
print(f"  Total children under 18: {total_children:,.0f}")
print(f"  Children under 6: {children_under_6:,.0f}")
print(f"  Children under 3: {children_under_3:,.0f}")


Children by age:
  Total children under 18: 1,157,115
  Children under 6: 350,517
  Children under 3: 177,575


In [7]:
# Create comprehensive summary table
summary_data = {
    'Metric': [
        'Household count (weighted)',
        'Person count (weighted)',
        'Average household size',
        'Weighted median household AGI',
        'Weighted average household AGI',
        'Weighted median person AGI',
        'Weighted average person AGI',
        'Unweighted median household AGI',
        'Unweighted median person AGI',
        '25th percentile household AGI',
        '75th percentile household AGI',
        '90th percentile household AGI',
        '95th percentile household AGI',
        'Max household AGI',
        'Total households with children',
        'Households with 1 child',
        'Households with 2 children',
        'Households with 3+ children',
        'Total children under 18',
        'Children under 6',
        'Children under 3'
    ],
    'Value': [
        f"{household_count.sum():,.0f}",
        f"{person_count.sum():,.0f}",
        f"{avg_hh_size:.1f}",
        f"${weighted_median_hh:,.0f}",
        f"${weighted_avg_hh:,.0f}",
        f"${weighted_median_person:,.0f}",
        f"${weighted_avg_person:,.0f}",
        f"${unweighted_median_hh:,.0f}",
        f"${unweighted_median_person:,.0f}",
        f"${weighted_percentile(agi_hh_array, hh_weights, 25):,.0f}",
        f"${weighted_percentile(agi_hh_array, hh_weights, 75):,.0f}",
        f"${weighted_percentile(agi_hh_array, hh_weights, 90):,.0f}",
        f"${weighted_percentile(agi_hh_array, hh_weights, 95):,.0f}",
        f"${agi_hh_array.max():,.0f}",
        f"{total_households_with_children:,.0f}",
        f"{households_with_1_child:,.0f}",
        f"{households_with_2_children:,.0f}",
        f"{households_with_3plus_children:,.0f}",
        f"{total_children:,.0f}",
        f"{children_under_6:,.0f}",
        f"{children_under_3:,.0f}"
    ]
}

summary_df = pd.DataFrame(summary_data)

print("\n" + "="*65)
print("SC STAGING DATASET SUMMARY - WEIGHTED (Population Estimates)")
print("="*65)
print(summary_df.to_string(index=False))
print("="*65)

# Save table
summary_df.to_csv('sc_staging_dataset_summary_weighted.csv', index=False)
print("\nSummary saved to: sc_staging_dataset_summary_weighted.csv")


SC STAGING DATASET SUMMARY - WEIGHTED (Population Estimates)
                         Metric        Value
     Household count (weighted)    1,587,912
        Person count (weighted)    4,788,894
         Average household size          3.0
  Weighted median household AGI      $60,378
 Weighted average household AGI     $105,617
     Weighted median person AGI      $56,366
    Weighted average person AGI     $100,583
Unweighted median household AGI      $67,907
   Unweighted median person AGI      $67,460
  25th percentile household AGI      $27,117
  75th percentile household AGI     $105,515
  90th percentile household AGI     $153,279
  95th percentile household AGI     $237,856
              Max household AGI $353,653,602
 Total households with children      637,558
        Households with 1 child      308,153
     Households with 2 children      190,106
    Households with 3+ children      139,299
        Total children under 18    1,157,115
               Children under 6      3

In [8]:
# Households with $0 income
agi_hh = np.array(sim.calculate("adjusted_gross_income", period=2025, map_to="household"))
weights = np.array(sim.calculate("household_weight", period=2025))

zero_income_mask = agi_hh == 0
zero_income_count = weights[zero_income_mask].sum()
total_households = weights.sum()

print("\n" + "="*70)
print("HOUSEHOLDS WITH $0 INCOME")
print("="*70)
print(f"Household count: {zero_income_count:,.0f}")
print(f"Percentage of all households: {zero_income_count / total_households * 100:.2f}%")
print("="*70)


HOUSEHOLDS WITH $0 INCOME
Household count: 9,513
Percentage of all households: 0.60%


In [9]:
# Household counts by income brackets
income_brackets = [
    (0, 10000, "$0-$10k"),
    (10000, 20000, "$10k-$20k"),
    (20000, 30000, "$20k-$30k"),
    (30000, 40000, "$30k-$40k"),
    (40000, 50000, "$40k-$50k"),
    (50000, 60000, "$50k-$60k")
]

bracket_data = []
for lower, upper, label in income_brackets:
    mask = (agi_hh >= lower) & (agi_hh < upper)
    count = weights[mask].sum()
    pct_of_total = (count / total_households) * 100
    
    bracket_data.append({
        "Income Bracket": label,
        "Households": f"{count:,.0f}",
        "% of All Households": f"{pct_of_total:.2f}%"
    })

income_df = pd.DataFrame(bracket_data)

print("\n" + "="*70)
print("HOUSEHOLD COUNTS BY INCOME BRACKET")
print("="*70)
print(income_df.to_string(index=False))
print("="*70)

# Total in $0-$60k range
total_in_range = sum([weights[(agi_hh >= lower) & (agi_hh < upper)].sum() for lower, upper, _ in income_brackets])
print(f"\nTotal households in $0-$60k range: {total_in_range:,.0f}")
print(f"Percentage of all households in $0-$60k range: {total_in_range / total_households * 100:.2f}%")


HOUSEHOLD COUNTS BY INCOME BRACKET
Income Bracket Households % of All Households
       $0-$10k    136,673               8.61%
     $10k-$20k    154,618               9.74%
     $20k-$30k    148,652               9.36%
     $30k-$40k    125,157               7.88%
     $40k-$50k    121,742               7.67%
     $50k-$60k    101,082               6.37%

Total households in $0-$60k range: 787,925
Percentage of all households in $0-$60k range: 49.62%
